# Threshold recalibration on Tufts and ADCD

Source: DENTEX. Targets: Tufts (Sections 3 to 7) and the ADCD external-validation split (Section 8).

The source threshold is the 95th percentile of the Mahalanobis distances of the DENTEX validation teeth (seed-42 radiograph-grouped split). The notebook reports:
1. Refusal of each target at the source threshold.
2. Refusal on held-out target crops after the threshold is reset to the 95th percentile of a random subset of unlabelled target crops, with 50 random draws per subset size.
3. For Tufts, the AUC of a per-tooth abnormality score on the accepted crops as the threshold is loosened. The radiograph-level normal or abnormal label serves as a coarse per-tooth reference. On ADCD this AUC is undefined because every ADCD crop is a positive condition box.

The ADCD embeddings are produced by `05_adcd_crops.ipynb` and loaded in Section 8.

## 1. Setup: paths, device and the frozen Dental ViT-L encoder

In [ ]:
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p],check=False)
# Packages are listed in requirements.txt. Uncomment to install them into the active kernel.
# pipq("torch","torchvision","scikit-learn","pillow","matplotlib","tqdm","huggingface_hub","ml_collections")
import torch, numpy as np, os, json, math, glob
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score
DEVICE="mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# Source dataset (DENTEX)
DATA_DIR  = "/path/to/DENTEX/training_data/quadrant-enumeration-disease"
# Target dataset (Tufts)
TUFTS_DIR = "/path/to/Tufts"
assert os.path.isdir(DATA_DIR),  f"DENTEX not found: {DATA_DIR}"
assert os.path.isdir(TUFTS_DIR), f"Tufts not found: {TUFTS_DIR}"

def find_dir(root, name):
    for r,_,_ in os.walk(root):
        if os.path.basename(r).lower()==name: return r
    return None
DX_IMG=find_dir(DATA_DIR,"xrays")
TF_IMG=None
for cand in ["radiographs","images"]:
    TF_IMG=TF_IMG or find_dir(TUFTS_DIR,cand)
OUT=os.path.join(DATA_DIR,"_results_recal"); os.makedirs(OUT,exist_ok=True)
CACHE=os.path.join(OUT,"cache"); os.makedirs(CACHE,exist_ok=True)
print("device:",DEVICE,"| DENTEX xrays:",DX_IMG,"| Tufts images:",TF_IMG)

In [ ]:
# dental ViT-L encoder (frozen) + crop/embed helpers
if not os.path.isdir("ViT-pytorch"):
    subprocess.run(["git","clone","--depth","1","https://github.com/jeonsworld/ViT-pytorch"],check=True)
sys.path.insert(0,"ViT-pytorch")
from models.modeling import VisionTransformer, CONFIGS
from huggingface_hub import hf_hub_download
ck=hf_hub_download(repo_id="Bryceee/Teeth_Visual_Experts_Models",
    filename="Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin")
state=torch.load(ck,map_location="cpu")
if isinstance(state,dict) and "state_dict" in state: state=state["state_dict"]
pos=state["transformer.embeddings.position_embeddings"].shape[1]
IMG=int(round(math.sqrt(pos-1)))*16; nh=state["head.weight"].shape[0]
m=VisionTransformer(CONFIGS["ViT-L_16"],img_size=IMG,num_classes=nh,zero_head=False)
m.load_state_dict(state,strict=False); m=m.to(DEVICE).eval()
import torchvision.transforms as T
prep=T.Compose([T.Resize((IMG,IMG)),T.ToTensor(),T.Normalize([0.5]*3,[0.5]*3)])
class DE(torch.nn.Module):
    def __init__(s,v): super().__init__(); s.v=v
    @torch.no_grad()
    def forward(s,x): seq,_=s.v.transformer(x); return seq[:,0]
enc=DE(m).to(DEVICE).eval()
def crop_box(img,box,pad=0.15,min_size=8):
    W,H=img.size; x0,y0,x1,y1=box; w,h=x1-x0,y1-y0
    if w<min_size or h<min_size: return None
    cx0,cy0=max(0,int(x0-w*pad)),max(0,int(y0-h*pad)); cx1,cy1=min(W,int(x1+w*pad)),min(H,int(y1+h*pad))
    if cx1-cx0<min_size or cy1-cy0<min_size: return None
    return img.crop((cx0,cy0,cx1,cy1))
@torch.no_grad()
def embed(pil):
    x=prep(pil.convert("RGB")).unsqueeze(0).to(DEVICE)
    return torch.nn.functional.normalize(enc(x).float().cpu(),dim=1).numpy()[0]
print("encoder ready @",IMG)

## 2. Source (DENTEX) embeddings and the Mahalanobis gate

In [ ]:
coco=None
for p in glob.glob(os.path.join(DATA_DIR,'**','*.json'),recursive=True):
    try: j=json.load(open(p))
    except Exception: continue
    if isinstance(j,dict) and "annotations" in j and any("category_id_3" in a for a in j.get("annotations",[])):
        coco=j; break
assert coco is not None, "DENTEX diagnosis json not found"
cats=sorted(coco["categories_3"],key=lambda c:c["id"]); DX_NAMES=[c["name"] for c in cats]
NUM=len(DX_NAMES); id2={c["id"]:i for i,c in enumerate(cats)}
imgf={im["id"]:im["file_name"] for im in coco["images"]}

emb_s=os.path.join(CACHE,"emb_dentex.npy"); lab_s=os.path.join(CACHE,"lab_dentex.npy"); img_s=os.path.join(CACHE,"imgof_dentex.npy")
if all(os.path.exists(p) for p in [emb_s,lab_s,img_s]):
    Xs=np.load(emb_s); Ys=np.load(lab_s); IMGs=np.load(img_s); print("loaded DENTEX cache:",Xs.shape)
else:
    tooth={}
    for a in coco["annotations"]:
        k=(a["image_id"],a.get("category_id_1",-1),a.get("category_id_2",-1))
        if k not in tooth: tooth[k]={"img":a["image_id"],"y":np.zeros(NUM,np.float32),"bbox":list(a["bbox"])}
        tooth[k]["y"][id2[a["category_id_3"]]]=1
        bx,by,bw,bh=tooth[k]["bbox"]; ax,ay,aw,ah=a["bbox"]
        x0,y0=min(bx,ax),min(by,ay); x1,y1=max(bx+bw,ax+aw),max(by+bh,ay+ah); tooth[k]["bbox"]=[x0,y0,x1-x0,y1-y0]
    rows=sorted(tooth.values(),key=lambda r:r["img"]); feats=[]
    cur=None; im=None
    for t in tqdm(rows):
        fp=os.path.join(DX_IMG,os.path.basename(imgf[t["img"]]))
        if fp!=cur: im=Image.open(fp).convert("RGB"); cur=fp
        x,y,w,h=t["bbox"]; feats.append(embed(im.crop((max(0,int(x-0.15*w)),max(0,int(y-0.15*h)),
                                                      min(im.size[0],int(x+w+0.15*w)),min(im.size[1],int(y+h+0.15*h))))))
    Xs=np.stack(feats).astype(np.float32); Ys=np.stack([t["y"] for t in rows]); IMGs=np.array([t["img"] for t in rows])
    np.save(emb_s,Xs); np.save(lab_s,Ys); np.save(img_s,IMGs); print("cached DENTEX:",Xs.shape)

# grouped split (train for gate statistics, val for the source threshold)
ids=sorted(set(IMGs.tolist())); rng=np.random.RandomState(42); rng.shuffle(ids)
n=len(ids); te=set(ids[:int(n*.15)]); va=set(ids[int(n*.15):int(n*.30)])
tr=[i for i,im in enumerate(IMGs) if im not in te and im not in va]
Xtr=Xs[tr]
mu=Xtr.mean(0); cov=np.cov(Xtr.T)+1e-2*np.eye(Xtr.shape[1]); cov_inv=np.linalg.inv(cov)
def maha(X): return np.einsum("ni,ij,nj->n",X-mu,cov_inv,X-mu)
tau_src=float(np.quantile(maha(Xs[[i for i,im in enumerate(IMGs) if im in va]]),0.95))
print(f"OOD gate fit | source 95th-pct threshold tau_src = {tau_src:.1f}")

## 3. Target (Tufts) tooth crops and image-level labels

`Segmentation/teeth_bbox.json` and `Expert/expert.json` are index-aligned lists of 1000 items. Each box is `[x1, y1, x2, y2]` on a portrait annotation canvas; the radiographs are stored upright landscape. Boxes are mapped with the extent-calibrated counter-clockwise transform (`nx = (ya - AY_MIN)/(AY_MAX - AY_MIN)`, `ny = (xa - AX_MIN)/(AX_MAX - AX_MIN)`, `image_x = nx*W`, `image_y = ny*H`) and checked visually in 3a before embedding. Image-level label: `Description == "Within normal limits"` is normal, anything else abnormal; every crop inherits its radiograph's label (coarse proxy).

In [ ]:
TF_BBOX_JSON   = os.path.join(TUFTS_DIR, "Segmentation", "teeth_bbox.json")
TF_EXPERT_JSON = os.path.join(TUFTS_DIR, "Expert", "expert.json")
bbox_items = json.load(open(TF_BBOX_JSON)); exp_items = json.load(open(TF_EXPERT_JSON))
assert len(bbox_items) == len(exp_items), "teeth_bbox.json and expert.json must be index-aligned"

def item_boxes(item):
    out=[]
    for o in (item.get("Label") or {}).get("objects",[]) or []:
        b=o.get("bounding box")
        if isinstance(b,(list,tuple)) and len(b)==4: out.append([float(v) for v in b])
    return out
ALL_BOXES=[b for it in bbox_items for b in item_boxes(it)]
AX_MIN=min(min(b[0],b[2]) for b in ALL_BOXES); AX_MAX=max(max(b[0],b[2]) for b in ALL_BOXES)
AY_MIN=min(min(b[1],b[3]) for b in ALL_BOXES); AY_MAX=max(max(b[1],b[3]) for b in ALL_BOXES)
print(f"Tufts boxes: {len(ALL_BOXES)} | extents x[{AX_MIN:.0f},{AX_MAX:.0f}] y[{AY_MIN:.0f},{AY_MAX:.0f}]")

def map_box(b,W,H):
    xs,ys=[],[]
    for xa,ya in ((b[0],b[1]),(b[2],b[3])):
        nx=(ya-AY_MIN)/(AY_MAX-AY_MIN); ny=(xa-AX_MIN)/(AX_MAX-AX_MIN)
        xs.append(nx*W); ys.append(ny*H)
    return [min(xs),min(ys),max(xs),max(ys)]

def tufts_path(ext_id):
    stem=os.path.splitext(ext_id)[0]
    for ext in (".jpg",".JPG",".jpeg",".png"):
        p=os.path.join(TF_IMG,stem+ext)
        if os.path.exists(p): return p
    return None

TUFTS_ITEMS=[]
for it_b,it_e in zip(bbox_items,exp_items):
    p=tufts_path(it_e.get("External ID","")); bxs=item_boxes(it_b)
    if p is None or not bxs: continue
    ab=0 if it_e.get("Description","").strip().lower()=="within normal limits" else 1
    TUFTS_ITEMS.append((p,bxs,ab))
print("Tufts radiographs with boxes:",len(TUFTS_ITEMS))

### 3a. Visual check of the mapped boxes

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
p,bxs,_=TUFTS_ITEMS[0]
im=Image.open(p).convert("RGB"); W,H=im.size
fig,ax=plt.subplots(figsize=(14,7)); ax.imshow(im)
for b in bxs:
    x0,y0,x1,y1=map_box(b,W,H)
    ax.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,edgecolor="lime",lw=1.2))
ax.set_title("CCW mapping calibrated to observed box extents: "+os.path.basename(p))
plt.tight_layout(); plt.show()

In [ ]:
emb_t=os.path.join(CACHE,"emb_tufts.npy"); lab_t=os.path.join(CACHE,"lab_tufts.npy"); imn_t=os.path.join(CACHE,"img_tufts.npy")
if all(os.path.exists(p) for p in [emb_t,lab_t,imn_t]):
    Xt=np.load(emb_t); yt=np.load(lab_t); IMGt=np.load(imn_t,allow_pickle=True); print("loaded Tufts cache:",Xt.shape)
else:
    feats=[]; labs=[]; names=[]; skipped=0
    for p,bxs,ab in tqdm(TUFTS_ITEMS):
        im=Image.open(p).convert("RGB"); W,H=im.size
        for b in bxs:
            c=crop_box(im,map_box(b,W,H))
            if c is None: skipped+=1; continue
            feats.append(embed(c)); labs.append(ab); names.append(os.path.basename(p))
    Xt=np.stack(feats).astype(np.float32); yt=np.array(labs); IMGt=np.array(names,dtype=object)
    np.save(emb_t,Xt); np.save(lab_t,yt); np.save(imn_t,IMGt)
    print(f"cached Tufts: {Xt.shape} | skipped {skipped}")
dist_t=maha(Xt)
print(f"Tufts teeth: {len(Xt)} | abnormal-image teeth: {int(yt.sum())} | normal: {int((yt==0).sum())}")

## 4. Tufts refusal at the source threshold

In [ ]:
refuse_baseline=float((dist_t>tau_src).mean())
print(f"Tufts refusal at the source 95th-percentile threshold: {refuse_baseline:.1%}")

## 5. Tufts: threshold recalibration with unlabelled target crops

The threshold is set from a random subset of Tufts crops, without labels, and the refusal rate is measured on the remaining crops; 50 random draws per subset size.

In [ ]:
def recal_refusal(n_cal, seed):
    r=np.random.RandomState(seed); idx=r.permutation(len(Xt)); cal,test=idx[:n_cal],idx[n_cal:]
    tau=np.quantile(dist_t[cal],0.95)
    return float((dist_t[test]>tau).mean()), float(tau)
rows=[]
for ncal in [10,25,50,100,200,400]:
    if ncal>=len(Xt): break
    rt=[recal_refusal(ncal,s)[0] for s in range(50)]
    rows.append((ncal,np.mean(rt),np.std(rt)))
    print(f"  unlabeled target crops={ncal:4d}  ->  refusal {np.mean(rt):.1%} +/- {np.std(rt):.1%}")
ns=[r[0] for r in rows]; mu_=[r[1]*100 for r in rows]; sd=[r[2]*100 for r in rows]
plt.figure(figsize=(6,3.4)); plt.errorbar(ns,mu_,yerr=sd,marker="o",color="#1a3a5c",capsize=3)
plt.axhline(refuse_baseline*100,ls="--",color="#a83232",label=f"source-threshold ({refuse_baseline:.0%})")
plt.xscale("log"); plt.xlabel("unlabeled target-center crops used to recalibrate")
plt.ylabel("refusal rate on held-out target (%)"); plt.title("Per-center threshold recalibration")
plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUT,"deployability.png"),dpi=200); plt.show()

## 6. Tufts: accepted-case AUC as the threshold is loosened

Logistic probes fitted on DENTEX give a per-tooth abnormality score (maximum over conditions). For each threshold, the AUC of that score against the radiograph-level Tufts label is computed on the accepted crops.

In [ ]:
from sklearn.linear_model import LogisticRegression
probes=[LogisticRegression(max_iter=1000).fit(Xtr,Ys[tr][:,c]) if len(np.unique(Ys[tr][:,c]))>1 else None
        for c in range(NUM)]
def probs(X):
    P=np.zeros((len(X),NUM))
    for c,p in enumerate(probes):
        P[:,c]=p.predict_proba(X)[:,1] if p is not None else 0.0
    return P
abn_score=probs(Xt).max(1)

sweep=[]
for pct in [40,50,60,70,80,90,95,99]:
    tau=np.quantile(dist_t,pct/100.0); acc=dist_t<=tau
    refusal=1-acc.mean()
    auc=roc_auc_score(yt[acc],abn_score[acc]) if (acc.sum()>40 and len(np.unique(yt[acc]))>1) else float("nan")
    sweep.append((refusal,acc.mean(),auc))
    print(f"  threshold {pct:>3d}%-of-target | refusal {refusal:5.1%} | accepted {acc.mean():5.1%} | accepted abnormality-AUC {auc:.3f}")

# reference on the source; guarded because DENTEX diagnosis teeth can be all-positive (single class)
y_src=(Ys[tr].max(1)>0).astype(int)
src_auc=roc_auc_score(y_src,probs(Xtr).max(1)) if len(np.unique(y_src))==2 else float("nan")
print(f"\n  reference: source (DENTEX-train) abnormality-AUC = {src_auc:.3f}")

ref=[s[0]*100 for s in sweep]; aucs=[s[2] for s in sweep]
plt.figure(figsize=(6,3.4))
plt.plot(ref,aucs,marker="o",color="#1a3a5c",label="accepted-case AUC (target)")
if not np.isnan(src_auc): plt.axhline(src_auc,ls="--",color="#3a7d44",label=f"source AUC ({src_auc:.2f})")
plt.axhline(0.5,ls=":",color="#a83232",label="chance (0.5)")
plt.xlabel("refusal rate on target (%)"); plt.ylabel("accepted-case abnormality AUC")
plt.title("Tufts: accepted-case AUC versus refusal")
plt.gca().invert_xaxis(); plt.legend(loc="upper right"); plt.tight_layout()
plt.savefig(os.path.join(OUT,"refusal_vs_quality.png"),dpi=200); plt.show()

# score spread on target vs source
sp_t=np.percentile(abn_score,[5,50,95]); sp_s=np.percentile(probs(Xtr).max(1),[5,50,95])
print(f"  Tufts score 5/50/95 pct: {sp_t.round(2)} | DENTEX-train: {sp_s.round(2)}")

## 7. Summary of the Tufts sweep

In [ ]:
usable=[s for s in sweep if s[0]<=0.30 and not np.isnan(s[2])]
print("="*70)
if usable and max(s[2] for s in usable) >= 0.65:
    op=max(usable,key=lambda s:s[2])
    print("Accepted-case AUC reaches 0.65 or more at a refusal rate of 30% or less.")
    print(f"  ~{op[0]:.0%} refusal, {op[1]:.0%} accepted, accepted-case AUC {op[2]:.2f}.")
else:
    print("Accepted-case AUC stays below 0.65 at every refusal rate of 30% or less.")
print("="*70)

## 8. ADCD external-validation split

Loads the Dental ViT-L ADCD condition-crop embeddings cached by `05_adcd_crops.ipynb`. `ADCD_CACHE` must point to the `_cache` folder written by that notebook.

In [ ]:
ADCD_CACHE = "/path/to/ADCD/panoramic_radiography_yolo_dataset_14_classes/_cache"
X_adcd = np.load(os.path.join(ADCD_CACHE, "emb_adcd_dental_vitl.npy"))
Y_adcd = np.load(os.path.join(ADCD_CACHE, "lab_adcd.npy"))
dist_adcd = maha(X_adcd)
print("ADCD loaded:", X_adcd.shape, "| baseline refusal at source threshold:",
      f"{(dist_adcd > tau_src).mean():.1%}")

In [ ]:
# Accepted-case reliability on ADCD. Expected to be undefined: every ADCD crop is a positive condition box.
P_adcd = probs(X_adcd)
score_adcd = P_adcd.max(1)
y_any = (Y_adcd.max(1) > 0).astype(int)

print("threshold | refusal | accepted | accepted abnormality-AUC | n_pos/n_neg")
for pct in [40,50,60,70,80,90,95,99]:
    tau = np.quantile(dist_adcd, pct/100.0)
    acc = dist_adcd <= tau
    refusal = 1 - acc.mean()
    ya = y_any[acc]
    if acc.sum() > 30 and len(np.unique(ya)) == 2:
        auc = roc_auc_score(ya, score_adcd[acc])
    else:
        auc = float('nan')
    npos, nneg = int(ya.sum()), int((ya==0).sum())
    print(f"  {pct:>3d}% | {refusal:5.1%} | {acc.mean():5.1%} | {auc:.3f} | {npos}/{nneg}")

In [ ]:
# ADCD deployability: threshold from a random subset of unlabeled ADCD crops, refusal on the rest (50 draws)
def adcd_recal_refusal(n_cal, seed):
    r = np.random.RandomState(seed)
    idx = r.permutation(len(dist_adcd))
    cal, test = idx[:n_cal], idx[n_cal:]
    tau = np.quantile(dist_adcd[cal], 0.95)
    return float((dist_adcd[test] > tau).mean())

print("ADCD per-center threshold recalibration:")
rows = []
for ncal in [10, 25, 50, 100, 200]:
    if ncal >= len(dist_adcd): break
    rt = [adcd_recal_refusal(ncal, s) for s in range(50)]
    rows.append((ncal, np.mean(rt), np.std(rt)))
    print(f"  unlabeled ADCD crops={ncal:4d} -> refusal {np.mean(rt):.1%} +/- {np.std(rt):.1%}")

ns  = [r[0] for r in rows]
mu_ = [r[1]*100 for r in rows]
sd  = [r[2]*100 for r in rows]
base = (dist_adcd > tau_src).mean()*100
plt.figure(figsize=(6,3.4))
plt.errorbar(ns, mu_, yerr=sd, marker="o", color="#1a3a5c", capsize=3, label="recalibrated (ADCD)")
plt.axhline(base, ls="--", color="#c9a227", lw=1.5, label=f"source threshold ({base:.0f}%)")
plt.xscale("log")
plt.xlabel("unlabeled ADCD crops used to recalibrate")
plt.ylabel("refusal rate on held-out ADCD (%)")
plt.title("ADCD (mild center shift): threshold recalibration")
plt.legend(loc="upper right", fontsize=8)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "adcd_deployability.png"), dpi=200)
plt.show()

In [ ]:
# Refusal at the same source threshold across centers (this notebook's tau_src)
print("Refusal at the source 95th-pct threshold (tau_src):")
print(f"  DENTEX (source)        : ~5% by construction")
print(f"  ADCD                   : {(dist_adcd > tau_src).mean():.1%}  (n={len(dist_adcd)})")
print(f"  Tufts                  : {(dist_t > tau_src).mean():.1%}  (n={len(dist_t)})")